In [1]:
print("Hello World")

Hello World


In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd

In [3]:
img_size = (224,224)
batch_size = 32

In [5]:
train = tf.keras.utils.image_dataset_from_directory(
    "./data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/train",
    image_size = img_size,
    batch_size = batch_size,
    shuffle = True,
    seed=42,
    label_mode = "categorical"
)

Found 6054 files belonging to 5 classes.


In [7]:
valid = tf.keras.utils.image_dataset_from_directory(
    "./data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/val",
    shuffle = False,
    image_size = img_size,
    batch_size = batch_size
)

Found 2016 files belonging to 5 classes.


In [8]:
test = tf.keras.utils.image_dataset_from_directory(
    "./data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/test",
    shuffle = False,
    image_size = img_size,
    batch_size = batch_size
)

Found 2025 files belonging to 5 classes.


In [20]:
from pathlib import Path

TRAIN_DIR = Path(r"./data\stage1_balanced\train")

for folder in sorted(TRAIN_DIR.iterdir()):
    if folder.is_dir():
        count = len([f for f in folder.iterdir() if f.is_file()])
        print(folder.name, ":", count)

Corona Virus Disease : 1218
Normal : 1207
Pneumonia : 1204
Tuberculosis : 1220


In [12]:
from pathlib import Path
import random
import shutil


SOURCE = Path(r"./data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/")
DEST = Path(r"C:/AI/Medical/Liver/data/lungs-disease-dataset-4-types/versions/2/Lung Disease Dataset/")

SEED = 42

random.seed(SEED)


def make_pneumonia_split(source_split, dest_split):

    bacterial_dir = source_split / "Bacterial Pneumonia"
    viral_dir = source_split / "Viral Pneumonia"

    pneumonia_dir = dest_split / "Pneumonia"
    pneumonia_dir.mkdir(parents=True, exist_ok=True)

    bacterial_files = [
        f for f in bacterial_dir.iterdir()
        if f.is_file()
    ]

    viral_files = [
        f for f in viral_dir.iterdir()
        if f.is_file()
    ]

    # Take equal number from both
    n = min(len(bacterial_files), len(viral_files))

    # Shuffle first
    random.shuffle(bacterial_files)
    random.shuffle(viral_files)

    bacterial_selected = bacterial_files[:n]
    viral_selected = viral_files[:n]

    # Copy bacterial
    for i, file in enumerate(bacterial_selected):
        destination = pneumonia_dir / f"bacterial_{i:04d}{file.suffix}"
        shutil.copy2(file, destination)

    # Copy viral
    for i, file in enumerate(viral_selected):
        destination = pneumonia_dir / f"viral_{i:04d}{file.suffix}"
        shutil.copy2(file, destination)

    print(f"{source_split.name}:")
    print(f"  Bacterial selected: {len(bacterial_selected)}")
    print(f"  Viral selected:     {len(viral_selected)}")
    print(f"  Total Pneumonia:    {len(bacterial_selected) + len(viral_selected)}")
    print()


for split in ["train", "val", "test"]:

    source_split = SOURCE / split
    dest_split = DEST / split

    dest_split.mkdir(parents=True, exist_ok=True)

    # Create Pneumonia
    make_pneumonia_split(source_split, dest_split)

    # Copy the other 3 classes
    for cls in [
        "Corona Virus Disease",
        "Normal",
        "Tuberculosis"
    ]:

        src = source_split / cls
        dst = dest_split / cls

        shutil.copytree(src, dst, dirs_exist_ok=True)

train:
  Bacterial selected: 1204
  Viral selected:     1204
  Total Pneumonia:    2408

val:
  Bacterial selected: 401
  Viral selected:     401
  Total Pneumonia:    802

test:
  Bacterial selected: 403
  Viral selected:     403
  Total Pneumonia:    806



In [19]:
from pathlib import Path
import random
import shutil

SOURCE = Path(
    r"C:/AI/Medical/Liver/data/lungs-disease-dataset-4-types/versions/2/Lung Disease Dataset"
)

DEST = Path(
    r"C:/AI/Medical/Liver/data/stage1_balanced"
)

random.seed(42)

for split in ["train", "val", "test"]:

    source_pneumonia = SOURCE / split / "Pneumonia"
    dest_pneumonia = DEST / split / "Pneumonia"

    # Create destination folder
    dest_pneumonia.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Separate bacterial/viral
    # -------------------------
    bacterial = [
        f for f in source_pneumonia.iterdir()
        if f.is_file() and f.name.lower().startswith("bacterial_")
    ]

    viral = [
        f for f in source_pneumonia.iterdir()
        if f.is_file() and f.name.lower().startswith("viral_")
    ]

    # Shuffle
    random.shuffle(bacterial)
    random.shuffle(viral)

    # Take HALF of each
    n_bacterial = len(bacterial) // 2
    n_viral = len(viral) // 2

    selected_bacterial = bacterial[:n_bacterial]
    selected_viral = viral[:n_viral]

    # -------------------------
    # Copy selected images
    # -------------------------
    for file in selected_bacterial:
        shutil.copy2(file, dest_pneumonia / file.name)

    for file in selected_viral:
        shutil.copy2(file, dest_pneumonia / file.name)

    print(f"\n{split}")
    print(f"Original bacterial: {len(bacterial)}")
    print(f"Selected bacterial: {len(selected_bacterial)}")
    print(f"Original viral:     {len(viral)}")
    print(f"Selected viral:     {len(selected_viral)}")
    print(f"Final Pneumonia:    {len(selected_bacterial) + len(selected_viral)}")

    # -------------------------
    # Copy the other 3 classes
    # -------------------------
    for cls in [
        "Corona Virus Disease",
        "Normal",
        "Tuberculosis"
    ]:
        shutil.copytree(
            SOURCE / split / cls,
            DEST / split / cls,
            dirs_exist_ok=True
        )


train
Original bacterial: 1204
Selected bacterial: 602
Original viral:     1204
Selected viral:     602
Final Pneumonia:    1204

val
Original bacterial: 401
Selected bacterial: 200
Original viral:     401
Selected viral:     200
Final Pneumonia:    400

test
Original bacterial: 403
Selected bacterial: 201
Original viral:     403
Selected viral:     201
Final Pneumonia:    402


In [21]:
from pathlib import Path

pneumonia_dir = Path(
    r"C:\AI\Medical\Liver\data\stage1_balanced\train\Pneumonia"
)

bacterial = list(pneumonia_dir.glob("bacterial_*"))
viral = list(pneumonia_dir.glob("viral_*"))

print("Bacterial:", len(bacterial))
print("Viral:", len(viral))
print("Total:", len(bacterial) + len(viral))

Bacterial: 602
Viral: 602
Total: 1204
